# 03 Demo: aspect-level sentiment for a hotel manager

Pipeline shown end to end: pasted reviews > mask personal details > split into sentences > tag aspects > aspect-aware sentiment (DeBERTa ABSA) > per-aspect summary with the source sentences.
**Colab:** run all cells, then open the link printed by the last cell (use a GPU runtime if available). After editing `src/`, Runtime > Restart session.

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.chdir('/content/drive/MyDrive/hotel-review-aspect-sentiment/notebooks')
except ImportError:
    pass

In [2]:
# !pip install -q gradio transformers sentencepiece     # uncomment if an import fails
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
import torch
from src import models as M
from src.demo import analyse_reviews, summarise, evidence, split_reviews, build_app, SAMPLE_REVIEWS, mask_personal_details

OUT = Path.cwd().parent / "outputs"; OUT.mkdir(exist_ok=True)
device = 0 if torch.cuda.is_available() else -1
absa = M.HFAbsaModel(device=device)

## 1. Walk through the pipeline on the sample reviews

In [3]:
print(mask_personal_details("Call me on +971 50 123 4567 or mail john.smith@example.com, see www.example.com"))
reviews = split_reviews(SAMPLE_REVIEWS)
table = analyse_reviews(reviews, absa)
table

Call me on [phone] or mail [email], see [link]


,review,sentence,aspect,sentiment
0,1,"The room was excellent, but the staff were rud...",staff,negative
1,1,"The room was excellent, but the staff were rud...",room,positive
2,2,Location was perfect and the breakfast was good.,location,positive
3,2,Location was perfect and the breakfast was good.,food,neutral
4,2,The wifi didn't work and the street was noisy ...,noise,negative
5,2,The wifi didn't work and the street was noisy ...,facilities,negative
6,3,Not a clean hotel at all.,cleanliness,negative
7,3,Bathroom was dirty and we were charged for thi...,cleanliness,negative
8,3,Bathroom was dirty and we were charged for thi...,room,negative
9,4,"Great value for the price, very comfy beds and...",room,positive


In [4]:
summary = summarise(table)
summary.to_csv(OUT / "demo_summary.csv", index=False)
table.to_csv(OUT / "demo_table.csv", index=False)
summary

sentiment,aspect,positive,neutral,negative,mentions,net_score
0,noise,0,0,1,1,-1.00
1,staff,0,0,1,1,-1.00
2,cleanliness,0,0,2,2,-1.00
3,facilities,0,1,1,2,-0.50
4,food,0,1,0,1,0.00
5,room,2,0,1,3,0.33
6,location,1,0,0,1,1.00
7,value,1,0,0,1,1.00


In [5]:
# Every number links back to source sentences (the report's answer to LLM hallucination risk).
for asp in summary["aspect"][:3]:
    print(asp, "->", evidence(table, asp, "negative"))

noise -> ["The wifi didn't work and the street was noisy at night."]
staff -> ['The room was excellent, but the staff were rude at check in.']
cleanliness -> ['Not a clean hotel at all.', "Bathroom was dirty and we were charged for things we didn't use."]


## 2. Why aspect-aware matters: one sentence, two verdicts
An aspect-blind model gives the whole sentence one label. The aspect-aware model is asked about each aspect separately.

In [6]:
tests = ["The room was excellent, but the staff were rude.",
         "The breakfast wasn't bad but the wifi didn't work.",
         "Not the cleanest hotel but the location is perfect and staff were helpful."]
rob = M.HFSentimentModel(device=device)
rows = analyse_reviews(tests, absa)
rows["aspect-blind RoBERTa"] = rob.predict(rows)
rows = rows.rename(columns={"sentiment": "aspect-aware ABSA"})
rows.to_csv(OUT / "demo_aspect_aware_vs_blind.csv", index=False)
pd.set_option("display.max_colwidth", 80)
rows[["sentence", "aspect", "aspect-aware ABSA", "aspect-blind RoBERTa"]]

,sentence,aspect,aspect-aware ABSA,aspect-blind RoBERTa
0,"The room was excellent, but the staff were rude.",staff,negative,positive
1,"The room was excellent, but the staff were rude.",room,positive,positive
2,The breakfast wasn't bad but the wifi didn't work.,food,positive,neutral
3,The breakfast wasn't bad but the wifi didn't work.,facilities,neutral,neutral
4,Not the cleanest hotel but the location is perfect and staff were helpful.,staff,positive,positive
5,Not the cleanest hotel but the location is perfect and staff were helpful.,location,positive,positive


## 3. Launch the web app
Opens a Gradio page (a public link appears when `share=True`). Paste reviews separated by blank lines.

In [ ]:
app = build_app(absa)
app.launch(share=True, debug=False)